# BitSwarm-RAG on Colab (T4)
Runs: real BGE embeddings, local LLM on GPU, eval table, 3-node swarm on localhost, attack suite.
**Not testable on Colab:** real LAN multicast (VM isolated). Prove that on your laptop with `scripts/spike_multicast.py`.

Runtime → Change runtime type → **T4 GPU**.

## 0. Setup (re-run after every VM reset)
Push this repo to GitHub, put the URL below. Persistent data lives on Drive.

In [ ]:
REPO = "https://github.com/YOUR_USER/bitswarm-rag.git"   # <-- change
from google.colab import drive; drive.mount("/content/drive")
import os, subprocess
os.makedirs("/content/drive/MyDrive/bitswarm-data", exist_ok=True)
if not os.path.exists("/content/bitswarm-rag"):
    subprocess.run(["git","clone",REPO,"/content/bitswarm-rag"],check=True)
else:
    subprocess.run(["git","-C","/content/bitswarm-rag","pull"],check=True)
%cd /content/bitswarm-rag
!pip install -q -e ".[bge,torrent,dev]"

## 1. Tests (mount gate, attacks, 3-node swarm, libtorrent)

In [ ]:
!python -m pytest tests -q 2>&1 | tail -8

## 2. Spike 1: libtorrent private swarm (DHT/PEX/LSD/trackers off)

In [ ]:
!python scripts/spike_libtorrent.py

## 3. Real embeddings (BGE-small) + eval table
Home dir is on fast local disk; copy to Drive at the end.

In [ ]:
import sys; sys.path.insert(0,"src")
from bitswarm.node import Node
from bitswarm.evaluate import evaluate, table
node = Node("/content/bs_home","colab",{"embedder":"bge","api_port":49901})
node.publish("eval/corpus","OS notes")
res = evaluate(node,"eval/questions.json")
print(table(res)); print(res["refusal"])

**Calibrate refusal cutoff:** look at `best_dense` on answerable vs unanswerable questions and set `embedder.refusal_cutoff` between them (do NOT keep the default guess).

In [ ]:
import numpy as np, json
qs=json.load(open("eval/questions.json"))
a=[node.ask(q["q"])["best_dense"] for q in qs["answerable"]]
u=[node.ask(q)["best_dense"] for q in qs["unanswerable"]]
print("answerable min/mean:",min(a),np.mean(a)); print("unanswerable max/mean:",max(u),np.mean(u))

## 4. Local LLM on the T4 (Ollama)
Extractive backend always works. For real synthesis start Ollama.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama","serve"]); time.sleep(5)
!ollama pull qwen2.5:7b

In [ ]:
node.backend = __import__("bitswarm.synth",fromlist=["x"]).OllamaBackend("qwen2.5:7b")
r = node.ask("Explain thrashing and how the working set model prevents it")
print(r["answer"]); print(r["warnings"]); print(r["provenance"])

In [ ]:
!python scripts/spike_llm_speed.py qwen2.5:7b   # record TTFT + tok/s for the README

## 5. Three-node swarm on localhost (C downloads from A and B)

In [ ]:
from pathlib import Path
A=Node("/content/sw/A","A",{"embedder":"bge","api_port":49911}); B=Node("/content/sw/B","B",{"embedder":"bge","api_port":49912}); C=Node("/content/sw/C","C",{"embedder":"bge","api_port":49913})
for n in (A,B,C): n.start(discovery=False, sync_interval=3600)
for n in (B,C): n.trust.add(A.identity.pubkey,"A")
B.peers.add_manual(A.addr()); C.peers.add_manual(A.addr()); C.peers.add_manual(B.addr())
sid=A.publish("eval/corpus","OS notes").name
B.sync_once(); B.accept(sid)
C.sync_once(); C.accept(sid)
t=C.transfers[sid]; print(t["state"], "from:", t["per_peer"], "| dl",t["dl_seconds"],"s | mount",t["mount_seconds"],"s")
print([ (s["step"],s["name"],s["ok"]) for s in t["steps"]])

## 6. Attack suite

In [ ]:
!python -m bitswarm.cli attack

## 7. UI in Colab (tunnel)

In [ ]:
!pip install -q uvicorn nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cf && chmod +x cf
import threading, uvicorn
from bitswarm.ui import create_app
threading.Thread(target=lambda: uvicorn.run(create_app(node),host="127.0.0.1",port=8080,log_level="warning"),daemon=True).start()
!./cf tunnel --url http://127.0.0.1:8080

## 8. Save to Drive before the VM dies

In [ ]:
!rsync -a /content/bs_home/ /content/drive/MyDrive/bitswarm-data/bs_home/
!cp eval/results/latest.json /content/drive/MyDrive/bitswarm-data/ 2>/dev/null || true